# RAG Aquila — Ingest sur Google Colab

**Avant de commencer** : active le GPU via `Runtime > Change runtime type > T4 GPU`

Lance les cellules dans l'ordre.

## Étape 1 — Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU disponible : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ERREUR : pas de GPU detecte.")
    print("Va dans Runtime > Change runtime type > T4 GPU et relance.")

## Étape 2 — Clone du projet + installation des dépendances

In [ ]:
# Clone le projet depuis GitHub (les PDFs ENS.pdf et SORBONNE.pdf sont inclus dans le repo)
!git clone https://github.com/paulhenriduranton-collab/RAG_Aquila.git

import os
os.chdir('/content/RAG_Aquila')
print("Dossier courant :", os.getcwd())
print("Documents disponibles :", os.listdir('documents/'))

In [ ]:
# Installation des dependances (~3-4 minutes)
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-text-splitters \
    langchain-huggingface \
    chromadb \
    sentence-transformers \
    rank_bm25 \
    pymupdf4llm \
    docx2txt \
    ftfy \
    transformers \
    accelerate

print("Installation terminee.")

## Étape 3 — Connexion à Google Drive

La base vectorielle sera sauvegardée sur Drive pour ne pas être perdue quand la session Colab ferme.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/vector_db_aquila', exist_ok=True)
print("Drive monte. La base sera sauvegardee dans : /content/drive/MyDrive/vector_db_aquila")

## Étape 4 — Remplacement d'Ollama par HuggingFace

Ollama est un serveur local — il ne tourne pas sur Colab.
On remplace `OllamaEmbeddings` et `OllamaLLM` par des équivalents HuggingFace qui utilisent le GPU.

- Embeddings : `BAAI/bge-m3` (meme modele qu'en local)
- LLM : `Qwen/Qwen2.5-1.5B-Instruct` (remplace `gemma2:2b`, taille similaire)

In [ ]:
import sys
from types import ModuleType
from pathlib import Path
import torch
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

device = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"

# Charge le modele d'embedding (meme modele qu'en local, mais via HuggingFace directement)
print("Chargement des embeddings bge-m3 (~570 MB)...")
_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device_str},
)
print("Embeddings OK.")

# Charge le LLM (utilise pour la contextualisation de chaque chunk)
print("Chargement du LLM Qwen2.5-1.5B (~3 GB)...")
_pipe = hf_pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=60,       # on veut juste une courte phrase de contexte
    do_sample=False,
    return_full_text=False,  # retourne uniquement les tokens generes, pas le prompt
    device=device,
)
_llm = HuggingFacePipeline(pipeline=_pipe)
print("LLM OK.")

# Cree un faux module langchain_ollama qui retourne nos objets HuggingFace
# quand ingest.py appelle OllamaEmbeddings(...) ou OllamaLLM(...)
class _FakeOllamaEmbeddings:
    def __new__(cls, model, **kwargs):
        return _embeddings  # retourne directement l'objet HuggingFaceEmbeddings

class _FakeOllamaLLM:
    def __new__(cls, **kwargs):
        return _llm  # retourne directement le HuggingFacePipeline

fake_ollama = ModuleType("langchain_ollama")
fake_ollama.OllamaEmbeddings = _FakeOllamaEmbeddings
fake_ollama.OllamaLLM = _FakeOllamaLLM
sys.modules["langchain_ollama"] = fake_ollama

print("Patch Ollama -> HuggingFace applique.")

## Étape 5 — Lancement de l'ingest

In [ ]:
import ingest
from pathlib import Path

# Redirige la base vectorielle vers Google Drive (persistance entre sessions)
ingest.VECTOR_DB_DIR = Path("/content/drive/MyDrive/vector_db_aquila")

print("Demarrage de l'ingest...")
ingest.main()
print("Ingest termine. La base est sauvegardee sur Drive.")

## Et ensuite ?

La base vectorielle est dans ton Google Drive (`vector_db_aquila/`).

Pour interroger le RAG depuis Colab, il faudra faire le meme patch Ollama -> HuggingFace dans `ask.py` (meme principe, cellule separee).